# Forecasting Optuna Search CV 
## Modulus Set 4

**Notebook Goal**
- A modeling pipeline that optimizes the hyperparameters of the sktime forecasters that have the [capavility:pred_int tag](https://www.sktime.net/en/stable/examples/01b_forecasting_proba.html) 
- This notebook will focus on the ones where `i mod 4 = 0` wher `i` is the index of the registry table in the above link.
- The work will be based on this documentation: [ForecastingOptunaSearchCV](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.model_selection.ForecastingOptunaSearchCV.html)

In [13]:
from sktime.forecasting.model_selection import (
    ForecastingOptunaSearchCV,
    )
from sktime.datasets import load_shampoo_sales
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
from sktime.forecasting.base import ForecastingHorizon
from sktime.split import ExpandingWindowSplitter
from sktime.split import temporal_train_test_split
import optuna
from  optuna.distributions import CategoricalDistribution, IntDistribution, FloatDistribution

In [14]:
from sktime.registry import all_estimators

models = all_estimators(
    "forecaster", filter_tags={"capability:pred_int": True}, as_dataframe=True
)

models = models.iloc[::4]
models.head()


,name,object
0,ARCH,<class 'sktime.forecasting.arch._uarch.ARCH'>
4,BATS,<class 'sktime.forecasting.bats.BATS'>
8,ConformalIntervals,<class 'sktime.forecasting.conformal.Conformal...
12,DirRecTimeSeriesRegressionForecaster,<class 'sktime.forecasting.compose._reduce.Dir...
16,EnbPIForecaster,<class 'sktime.forecasting.enbpi.EnbPIForecast...


In [15]:
y = load_shampoo_sales()
y_train, y_test = temporal_train_test_split(y=y, test_size=6)
fh = ForecastingHorizon(y_test.index, is_relative=False).to_relative(cutoff=y_train.index[-1])
cv = ExpandingWindowSplitter(fh=fh, initial_window=24, step_length=1)
forecaster = models.iloc[0][1]()

In [16]:
param_grid = {
    "mean": CategoricalDistribution(["Constant", "Zero", "LS", "AR", "ARX", "HAR", "HARX"]),
    "lags": CategoricalDistribution([0, 1, 2, [1, 2], [1, 3, 5]]),  # Single or multiple lags
    "vol": CategoricalDistribution(["GARCH", "ARCH", "EGARCH", "FIARCH", "HARCH"]),
    # "p": IntDistribution(1, 5),  # Order of symmetric innovation
    # "o": IntDistribution(0, 3),  # Order of asymmetric innovation
    # "q": IntDistribution(1, 5),  # Order of lagged volatility
    # "power": FloatDistribution(1.0, 3.0),  # Power transformation
    "dist": CategoricalDistribution(["normal", "t", "skewstudent", "ged"]),  # Error distribution
    "rescale": CategoricalDistribution([True, False]),  # Whether to rescale data
    "cov_type": CategoricalDistribution(["robust", "classic"]),  # Covariance estimation method
    "method": CategoricalDistribution(["analytic", "simulation", "bootstrap"]),  # Forecasting method
    "simulations": IntDistribution(10, 500),  # Number of simulations (if using simulation/bootstrap)
    "align": CategoricalDistribution(["origin", "target"]),  # How forecasts are aligned
    "reindex": CategoricalDistribution([True, False]),  # Whether to reindex forecasts
}

/home/julia/projects/nocturnal-hypo-gly-prob-forecast/venv/lib/python3.10/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [1, 2] which is of type list.
  warnings.warn(message)
/home/julia/projects/nocturnal-hypo-gly-prob-forecast/venv/lib/python3.10/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [1, 3, 5] which is of type list.
  warnings.warn(message)


In [17]:
gscv = ForecastingOptunaSearchCV(
    forecaster=forecaster,
    param_grid=param_grid,
    cv=cv,
    n_evals=10,
    )
gscv.fit(y)
print(f"{gscv.best_params_=}")

/home/julia/projects/nocturnal-hypo-gly-prob-forecast/venv/lib/python3.10/site-packages/sktime/forecasting/model_selection/_tune.py:1773: UserWarning: ForecastingOptunaSearchCV is experimental, and interfaces may change. User feedback and suggestions for future development are appreciated in issue #6618 here: https://github.com/sktime/sktime/issues/6618
  warn(
[I 2025-02-06 17:42:49,159] A new study created in memory with name: no-name-df1603a6-46cf-4b75-b336-7bdb28012af5
/home/julia/projects/nocturnal-hypo-gly-prob-forecast/venv/lib/python3.10/site-packages/sktime/utils/parallel.py:92: FitFailedWarning: 
                In evaluate, fitting of forecaster ARCH failed,
                you can set error_score='raise' in evaluate to see
                the exception message.
                Fit failed for the 0-th data split, on training data y_train with
                cutoff NaT, and len(y_train)=24.
                The score will be set to nan.
                Failed forecaster with 

gscv.best_params_={'mean': 'LS', 'lags': 1, 'vol': 'ARCH', 'dist': 't', 'rescale': False, 'cov_type': 'robust', 'method': 'simulation', 'simulations': 464, 'align': 'origin', 'reindex': True}


/home/julia/projects/nocturnal-hypo-gly-prob-forecast/venv/lib/python3.10/site-packages/sktime/utils/parallel.py:92: FitFailedWarning: 
                In evaluate, fitting of forecaster ARCH failed,
                you can set error_score='raise' in evaluate to see
                the exception message.
                Fit failed for the 6-th data split, on training data y_train with
                cutoff NaT, and len(y_train)=30.
                The score will be set to nan.
                Failed forecaster with parameters: ARCH(align='target', cov_type='classic', dist='skewstudent', lags=[1, 2],
     mean='AR', method='analytic', reindex=True, simulations=398, vol='HARCH').
                
  ret = [fun(x, meta=meta) for x in iter]
/home/julia/projects/nocturnal-hypo-gly-prob-forecast/venv/lib/python3.10/site-packages/optuna/study/_tell.py:164: UserWarning: The value nan is not acceptable
  warnings.warn(values_conversion_failure_message)
